In [1]:
from delta.tables import DeltaTable

StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 3, Finished, Available, Finished, False)

In [2]:
df_silver = spark.read.format("delta").load("abfss://E_commerce@onelake.dfs.fabric.microsoft.com/lakehouse.Lakehouse/Tables/silver_transactions")


StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 4, Finished, Available, Finished, True)

In [3]:
import pyspark.sql.functions as f

StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 5, Finished, Available, Finished, True)

In [4]:
dim_product = df_silver.groupBy("ProductNo").agg(
    f.first("ProductName").alias("ProductName"),
    f.count("TransactionNo").alias("Total_Transactions"),
    f.round(f.avg("price"),2).alias("avg_price")
)


display(dim_product.limit(5))

StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bf179307-9acd-449a-81d4-98e5030aeaf1)

In [5]:
dim_product.write.format("delta").saveAsTable("Dim_products")

StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 7, Finished, Available, Finished, False)

In [6]:
dim_customer = df_silver.groupBy("CustomerNo").agg(
    f.last("country").alias("country")
)


display(dim_customer.limit(5))

StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e1e49c6f-cc8c-4041-8977-e6264f79b659)

In [7]:
dim_customer.write.format("delta").saveAsTable("Dim_customer")

StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 9, Finished, Available, Finished, False)

In [8]:
dim_date = df_silver.select("Date").distinct()\
.withColumn("Year",f.year("Date"))\
.withColumn("Month",f.month("Date"))\
.withColumn("Day",f.dayofmonth("Date"))\
.withColumn("Qurater",f.quarter("Date"))\
.withColumn("Day_of_week",f.dayofweek("Date"))\
.withColumn("Is_weekend",f.when(f.col("Day_of_week").isin(1,7),1).otherwise(0))


display(dim_date.limit(5))

StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38ee6feb-1c19-453a-a39f-e7916a6f8d16)

In [9]:
dim_date.write.format("delta").saveAsTable("Dim_Date")

StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 11, Finished, Available, Finished, False)

In [10]:
fact_table = df_silver.select(
    f.col("TransactionNo").alias("transaction_no"),
    f.col("ProductNo").alias("product_no"),
    f.col("CustomerNo").alias("cutomer_no"),
    f.col("Date").alias("date_key"),
    f.col("Price").alias("price"),
    f.col("Revenu").alias("revenu"),
    f.col("ReturnFlag").alias("return_flag")
)


display(fact_table.limit(5))

StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 258fd2b5-1406-4f38-815a-4af1038aa82c)

In [11]:
fact_table.write.format("delta").saveAsTable("fact_transactions")

StatementMeta(, c9439b60-7854-4cda-9cd7-88e14e5ce88f, 13, Finished, Available, Finished, False)